In [23]:
import pandas as pd
import numpy as np

In [36]:
df_imp = pd.read_csv(r"dataset\URA_renting_imputed_saits_no_cutoff_macro.csv")
df_imp = df_imp[["date", "condo_name", "size_tier", "rent_psf_imp"]]
df_imp

,date,condo_name,size_tier,rent_psf_imp
0,1999-11,1 MOULMEIN RISE,SZ3,3.769841
1,1999-12,1 MOULMEIN RISE,SZ3,3.771444
2,2000-01,1 MOULMEIN RISE,SZ3,3.832254
3,2000-02,1 MOULMEIN RISE,SZ3,3.941147
4,2000-03,1 MOULMEIN RISE,SZ3,4.054842
...,...,...,...,...
711580,2025-09,ZENITH,SZ5,3.326864
711581,2025-10,ZENITH,SZ5,3.314321
711582,2025-11,ZENITH,SZ5,3.300587
711583,2025-12,ZENITH,SZ5,3.274982


In [37]:
df_ccr = pd.read_csv(r"dataset\database_v3\Graph_Size\node_id_ccr.csv")
df_ccr

,node_id,project_id,Project Name,size_bucket,node_key,count
0,10,3,1 MOULMEIN RISE,SZ3,3 | SZ3,67
1,11,3,1 MOULMEIN RISE,SZ4,3 | SZ4,82
2,12,4,1 NASSIM,SZ5,4 | SZ5,65
3,13,5,10 EVELYN,SZ1,5 | SZ1,15
4,14,5,10 EVELYN,SZ2,5 | SZ2,16
...,...,...,...,...,...,...
2309,7950,2495,ZENITH,SZ1,2495 | SZ1,299
2310,7951,2495,ZENITH,SZ2,2495 | SZ2,215
2311,7952,2495,ZENITH,SZ3,2495 | SZ3,28
2312,7953,2495,ZENITH,SZ4,2495 | SZ4,27


In [43]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
# Change this to your actual timestep folder
TIMESTEP_DIR = Path("dataset/database_v3/Graph_Size/timestep")

keep_cols = [
    "timestep",
    "node_id",
    "project_id",
    "Project Name",
    "size_bucket",
    "rent_per_sqft",
    "y_mask",
]

all_dfs = []

for csv_path in tqdm(sorted(TIMESTEP_DIR.glob("*.csv")), desc="Loading timestep CSVs"):
    df = pd.read_csv(csv_path)

    # Extract date from filename, e.g. 1_1999-12.csv -> 1999-12
    date_str = csv_path.stem.split("_", 1)[1]

    missing_cols = [col for col in keep_cols if col not in df.columns]
    if missing_cols:
        print(f"Skip {csv_path.name}: missing columns {missing_cols}")
        continue

    df = df[keep_cols].copy()
    df["date"] = date_str

    all_dfs.append(df)

big_df = pd.concat(all_dfs, ignore_index=True)

print(big_df.shape)
print(big_df.head())


Loading timestep CSVs:   0%|          | 0/315 [00:00<?, ?it/s]

Loading timestep CSVs: 100%|██████████| 315/315 [00:19<00:00, 16.26it/s]

(1229470, 8)
   timestep  node_id  project_id      Project Name size_bucket  rent_per_sqft  \
0         0      843       276.0       CANNE LODGE         SZ2       1.882353   
1         0     2063       651.0    FAR EAST PLAZA         SZ1       3.072727   
2         0     3506      1139.0  MANDARIN GARDENS         SZ2       2.760196   
3         0     4754      1528.0       REGENT PARK         SZ3       2.173913   
4         0     6877      2163.0         THE PLAZA         SZ2       3.530667   

   y_mask     date  
0       1  1999-11  
1       1  1999-11  
2       1  1999-11  
3       1  1999-11  
4       1  1999-11  


In [44]:
big_df = (
    big_df
    .sort_values(["node_id", "timestep"])
    .reset_index(drop=True)
)

print(big_df.shape)
print(big_df.head())

(1229470, 8)
   timestep  node_id  project_id Project Name size_bucket  rent_per_sqft  \
0       197        0         0.0     # 1 LOFT         SZ1       4.123457   
1       198        0         0.0     # 1 LOFT         SZ1       4.179574   
2       199        0         0.0     # 1 LOFT         SZ1       4.361111   
3       200        0         0.0     # 1 LOFT         SZ1       4.333333   
4       201        0         0.0     # 1 LOFT         SZ1       4.111111   

   y_mask     date  
0       1  2016-04  
1       1  2016-05  
2       1  2016-06  
3       1  2016-07  
4       1  2016-08  


In [45]:
merged_df = big_df.merge(
    df_imp,
    how="left",
    left_on=["date", "Project Name", "size_bucket"],
    right_on=["date", "condo_name", "size_tier"],
)
merged_df = merged_df.drop(columns=["condo_name", "size_tier"])

print(merged_df.shape)
print(merged_df.head())

(1229470, 9)
   timestep  node_id  project_id Project Name size_bucket  rent_per_sqft  \
0       197        0         0.0     # 1 LOFT         SZ1       4.123457   
1       198        0         0.0     # 1 LOFT         SZ1       4.179574   
2       199        0         0.0     # 1 LOFT         SZ1       4.361111   
3       200        0         0.0     # 1 LOFT         SZ1       4.333333   
4       201        0         0.0     # 1 LOFT         SZ1       4.111111   

   y_mask     date  rent_psf_imp  
0       1  2016-04           NaN  
1       1  2016-05           NaN  
2       1  2016-06           NaN  
3       1  2016-07           NaN  
4       1  2016-08           NaN  


In [46]:
import numpy as np

merged_df["rent_merged"] = np.where(
    merged_df["rent_psf_imp"].isna(),
    merged_df["rent_per_sqft"],
    np.where(
        merged_df["y_mask"] == 0,
        merged_df["rent_psf_imp"],
        merged_df["rent_per_sqft"],
    )
)

print(merged_df[[
    "rent_per_sqft",
    "rent_psf_imp",
    "y_mask",
    "rent_merged"
]].head())

   rent_per_sqft  rent_psf_imp  y_mask  rent_merged
0       4.123457           NaN       1     4.123457
1       4.179574           NaN       1     4.179574
2       4.361111           NaN       1     4.361111
3       4.333333           NaN       1     4.333333
4       4.111111           NaN       1     4.111111


In [47]:
merged_df.to_csv("dataset/database_v3/Graph_Size/merged_imputed_data.csv", index=False)